# General Tips

Here are some things I do that should help regardless of what you are doing. 

## Quickly Finding Data

I've seen students submit absolute paths to read data into memory with things like 

`df = pd.read_csv(r"C:\Users\Administrator\Downloads\Reviews.csv")`

While this works for you, it displays your computer's internal structure and doesn't let me run the files. A better way to obsfucate and quickly define your files is to rely on the Python module *os*. 

In [ ]:
import os
import pandas as pd

base_directory = os.getcwd() # This gets the path of where your Jupyter Notebook is running
data_file_path = os.path.join(base_directory, 'data_file_name.csv') # Defines your data file path as the same folder and adds the file name
df = pd.read_csv(data_file_path)

Beyond hiding what your directory looks like, this is compatiable with any OS you (or anyone else) use. It also lets you use relative pathing to get data. 

Let's pretend that I have organized my computer so that I have a folder for this course somewhere on the computer, and I put all of the data within a folder of that folder called "Data". To load that data, I can use:

In [ ]:
import os
import pandas as pd

# Read this as "get current working directory (cwd)"
base_directory = os.getcwd() 

# Defines your data file path as:
#   starting from the folder the Jupyter Notebook is in
#   look in the folder here called 'Data'
#   look for the file here called 'data_file_name.csv'
data_file_path = os.path.join(base_directory, 'Data', 'data_file_name.csv')

df = pd.read_csv(data_file_path)

## Displaying Variables in a String

There are a few ways to display variables in a string. My suggested method would be to use f-strings. Modern Python suggests using f-strings as well, though if you choose one and are consistent it isn't a problem. 

Here's an example comparison between f-strings and other methods:

In [21]:
import random
example_variable = random.randint(0, 10)

# f-string
# define f"" as an f-string. You can now insert variables where you want within braces
# You can manipulate your variables within these braces!
print(f'The example variable is: {example_variable}. It is 3 less than {example_variable+3}.\n')

# Adding strings
# We can add strings with the plus sign. So if your value is a string, you can directly add it
# And if it isn't a string (like in our example), wrap it in str() to make it a string
print('The example variable is: ' + str(example_variable) + '. It is 3 less than ' + str(example_variable+3) + '.')
print('Notice there is no space between each string added.\n')

# Commas in print statement
# A print statement will take any length of arguments and combine them into a single string, much like the adding string example above
# This will add a space between each argument though!
print('The example variable is:', example_variable, '. It is 3 less than', example_variable+3, '.')
print('Notice there is a space between each argument.\n')

# Format method (older Python version)
# A predecessor to f-strings where you leave empty braces and define your variables to fit in the spots afterwards
print('The example variable is: {}. It is 3 less than {}.\n'.format(example_variable, example_variable+3))

# % operator (C-style formatting)
# If you are more familiar with C-style programming, you can use that notation
# %i for integer, %s for string, %f for float, and others
print('The example variable is: %i. It is 3 less than %i.\n' %(example_variable, example_variable+3))

The example variable is: 8. It is 3 less than 11.

The example variable is: 8. It is 3 less than 11.
Notice there is no space between each string added.

The example variable is: 8 . It is 3 less than 11 .
Notice there is a space between each argument.

The example variable is: 8. It is 3 less than 11.

The example variable is: 8. It is 3 less than 11.



## Using DataFrames to Display Info

While a series of print statements can be used, you'll find that saving this information to a DataFrame for display will be more useful in the future. Not only will you have the data to bring forward and manipulate, but it looks better in a Jupyter Notebook file. 

### Code for Example

In [8]:
# This example is from: https://github.com/Darryl-Chamberlain-Jr/data_science_with_python/blob/main/2_Data%20Science%20with%20Python/Overview%20Notes/4_model_selection.ipynb
import random

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC 
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

import pandas as pd

from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()

# We use 'columns=' to define the names of the columns from the original dataset
predictor_df = pd.DataFrame(data['data'], columns=data['feature_names'])
response_df = pd.DataFrame(data['target'], columns=['target_class'])

def run_random_forest_classificiation(X_train, y_train, fixed_random_state):
    clf_rf = RandomForestClassifier(random_state=fixed_random_state, class_weight='balanced_subsample')
    clf_rf.fit(X_train, y_train)
    return clf_rf

def run_support_vector_classification(X_train, y_train, fixed_random_state):
    # Note LinearSVC scales better to large numbers of samples
    # We could also standarize the data before running SVC, which would improve speed and maybe accuracy
    clf_svc = LinearSVC(random_state=fixed_random_state, class_weight='balanced')
    clf_svc.fit(X_train, y_train)
    return clf_svc

def run_logit_classification(X_train, y_train, fixed_random_state):
    # Need to increase max iterations to converge to a solution
    clf_logit = LogisticRegression(random_state=fixed_random_state, max_iter=10000, class_weight='balanced')
    clf_logit.fit(X_train, y_train)
    return clf_logit

def evaluate_classification_model(clf_model, X_test, y_test):
    predictions = clf_model.predict(X_test)

    eval_dict = {
        'model_name': clf_model.__class__.__name__,
        'accuracy': accuracy_score(y_test, predictions),
        'recall': recall_score(y_test, predictions),
        'precision': precision_score(y_test, predictions),
        'f1_score': f1_score(y_test, predictions)
    }

    return eval_dict

def evaluate_three_classification_models(X, y):
    fixed_random_state = random.randint(0, 200)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=fixed_random_state)

    # y_train.values.ravel() used to fix an unexpected "DataConversionWarning" warning
    clf_rf = run_random_forest_classificiation(X_train, y_train.values.ravel(), fixed_random_state)
    rf_eval_dict = evaluate_classification_model(clf_rf, X_test, y_test)

    clf_svc = run_support_vector_classification(X_train, y_train.values.ravel(), fixed_random_state)
    svc_eval_dict = evaluate_classification_model(clf_svc, X_test, y_test)

    clf_logit = run_logit_classification(X_train, y_train.values.ravel(), fixed_random_state)
    logit_eval_dict = evaluate_classification_model(clf_logit, X_test, y_test)

    eval_df = pd.DataFrame([rf_eval_dict, svc_eval_dict, logit_eval_dict])
    return eval_df

def print_from_dict(eval_dict):
    print(f"\n Model name: {eval_dict['model_name']}")
    print(f"Accuracy: {eval_dict['accuracy']}")
    print(f"Recall: {eval_dict['recall']}")
    print(f"Precision: {eval_dict['precision']}")
    print(f"F1 Score: {eval_dict['f1_score']}")

def evaluate_three_classification_models_print_statements(X, y):
    fixed_random_state = random.randint(0, 200)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=fixed_random_state)

    # y_train.values.ravel() used to fix an unexpected "DataConversionWarning" warning
    clf_rf = run_random_forest_classificiation(X_train, y_train.values.ravel(), fixed_random_state)
    rf_eval_dict = evaluate_classification_model(clf_rf, X_test, y_test)
    print_from_dict(rf_eval_dict)

    clf_svc = run_support_vector_classification(X_train, y_train.values.ravel(), fixed_random_state)
    svc_eval_dict = evaluate_classification_model(clf_svc, X_test, y_test)
    print_from_dict(svc_eval_dict)

    clf_logit = run_logit_classification(X_train, y_train.values.ravel(), fixed_random_state)
    logit_eval_dict = evaluate_classification_model(clf_logit, X_test, y_test)
    print_from_dict(logit_eval_dict)

### Print Statements to Display Info

In [9]:
evaluate_three_classification_models_print_statements(predictor_df, response_df)


 Model name: RandomForestClassifier
Accuracy: 0.9736842105263158
Recall: 0.9868421052631579
Precision: 0.974025974025974
F1 Score: 0.9803921568627451

 Model name: LinearSVC
Accuracy: 0.9912280701754386
Recall: 0.9868421052631579
Precision: 1.0
F1 Score: 0.9933774834437086

 Model name: LogisticRegression
Accuracy: 0.9824561403508771
Recall: 0.9868421052631579
Precision: 0.9868421052631579
F1 Score: 0.9868421052631579


### DataFrame to Display Info

In [12]:
eval_df = evaluate_three_classification_models(predictor_df, response_df)
eval_df.round(3) # Since the evaluation data is stored in a dataframe, we can easily manipulate it

,model_name,accuracy,recall,precision,f1_score
0,RandomForestClassifier,0.947,0.975,0.951,0.963
1,LinearSVC,0.947,0.975,0.951,0.963
2,LogisticRegression,0.930,0.962,0.939,0.951


## Create Arrays to Test Values

Let's say we have a choice on some cutoff. We can create a function to iterate and test different points to figure out the "best" cutoff point. 

In [15]:
import numpy as np

arr = np.linspace(0.8, 1.0, 21)
arr

array([0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87, 0.88, 0.89, 0.9 ,
       0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 1.  ])

Linspace is suggested with fractional steps. Since linspace requires knowing how many values you need for the step size you want, you can create a quick function to give you the value instead.

In [20]:
import numpy as np

def create_list_with_step_size(start_value, end_value, wanted_step_size):
    # Divide the length of the interval by step size, then add 1 since you want both endpoints
    number_of_points = int( round((end_value - start_value) / wanted_step_size) ) + 1
    arr = np.linspace(start_value, end_value, number_of_points)
    return arr

In [21]:
create_list_with_step_size(0.8, 1, 0.01)

array([0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87, 0.88, 0.89, 0.9 ,
       0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 1.  ])

Now we can create a loop.

In [ ]:
for possible_cutoff in create_list_with_step_size(0.8, 1, 0.01):
    # Run your model here using the value possible_cutoff